## Open Tools, Open Sky
### Doing LSST Science with Public Rubin Data & LINCC Frameworks
#### Latest update: Jorge I. Zuluaga

## Template notebook

In [1]:
from util import *
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import alerce

# Macros
DEG = np.pi / 180
RAD = 1 / DEG
ecl_coords, gal_coords = ecl_gal()

# Directories
!mkdir -p data figures
fig_dir='figures/'
data_dir='data/'

# Reload util.py routines
%load_ext autoreload 
%autoreload 2

- **tap**: table application protocol
- **dia**: difference image analysis 

In [3]:
import pyvo as vo
from alerce.core import Alerce

alerce = Alerce()

tap_service_url = "https://tap.alerce.online/tap"
service = vo.dal.TAPService(tap_service_url)

# service.search() posts to /sync. TOP 200000 runs past the gateway
# timeout: HTTP 502, which pyvo reports as E19 because the error page
# is not a VOTable. This selection has fewer than 10000 rows.
query = """
SELECT DISTINCT TOP 10000
    obj.oid, obj.ra, obj.dec, obj.ndiasources, tax.class_name, prob.probability
FROM alerce_tap.lsst_dia_object AS obj
JOIN alerce_tap.probability AS prob ON obj.oid = prob.oid
JOIN alerce_tap.taxonomy AS tax ON prob.class_id = tax.class_id
WHERE tax.class_name = 'SN'
    AND prob.probability >= 0.8 AND prob.ranking=1
    AND obj.ndiasources > 1 AND tax.classifier_id=2
"""

# Synchronous query. The async endpoint at this service does not return.
df_vs = service.run_sync(query).to_table().to_pandas()
df_vs


,oid,ra,dec,ndiasources,class_name,probability
0,170019695992242795,52.823847,-29.691600,7,SN,0.991934
1,170019695992242795,52.823847,-29.691600,7,SN,0.994920
2,170019696008495213,53.169752,-29.185676,13,SN,0.974966
3,170019696046768167,54.237795,-27.670760,7,SN,0.910110
4,170019696046768167,54.237795,-27.670760,7,SN,0.973086
...,...,...,...,...,...,...
7971,314051320318263356,56.862116,-48.948286,123,SN,0.889471
7972,314051320443043860,62.669726,-47.693403,62,SN,0.916036
7973,314051320803229791,51.869484,-28.895136,9,SN,0.987434
7974,314051320940068947,52.588064,-28.315550,8,SN,0.830271
